# Let's make a 3-way conversation

The conversation will be between GPT-5-nano, Gemini-3.1-flash-lite-preview and local llama3.2, 
We're using cheap versions of models so the costs will be minimal.


In [ ]:
# Imports
import os
import requests
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display


In [ ]:
# Load environment
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set")


In [ ]:
# Connect to OpenAI client library
# A thin wrapper around calls to HTTP endpoints
openai = OpenAI()

# For Gemini and Ollama we can use the OpenAI python client
# Because Google and Ollama have endpoints compatible with OpenAI
# And OpenAI allows you to change the base_url
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
ollama_url = "http://localhost:11434/v1"

gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)

# Models
gpt_model = "gpt-5-nano"
gemini_model = "gemini-3.1-flash-lite-preview"
ollama_model = "llama3.2"

# System messages
gpt_system = """
    You are Alex, a JS Developer.
    You are in a conversation with Blake and Charlie.
    You always ask what the Ruby code does to Blake.
    """

gemini_system = """
    You are Blake, a Ruby Developer and you are happy to explain Ruby concepts to beginners.
    You are in a conversation with Alex and Charlie.
"""

ollama_system = """
    You are Charlie, a Fullstack Developer and provide code snippets examples
    when anyone talks about the Ruby programming language.
    You are in a conversation with Alex and Blake.
"""

# Initial messages
gpt_messages = ["""
    Hello, nice to meet you! I'm Alex and I want to learn more about Ruby.
"""]
gemini_messages = ["""
    Hello, nice to meet you too! I'm Blake and I'm a Ruby expert.
    Could you provide a code snippet so that I can explain it to Alex?
"""]
ollama_messages = ["""
    Hello, nice to meet you too! I'm Charlie and I can provide a new code snippet everytime you ask.
    How about we start a simple 'Hello World' example:
    ```ruby
    puts "Hello, World!"
    ```
"""]


In [ ]:
# Functions that build the complete conversation context

def call_gpt():
    messages = [{"role": "system", "content": gpt_system}]
    conversation = ""
    for gpt, gemini_message, ollama_message in zip(gpt_messages, gemini_messages, ollama_messages):
        if gpt and gemini and ollama:
            conversation = conversation + f"### GPT:\n{gpt}\n"
            conversation = conversation + f"### Gemini:\n{gemini_message}\n"
            conversation = conversation + f"### Ollama:\n{ollama_message}\n"
    user_prompt = f"""
        You are Alex, in a conversation with Blake and Charlie.
        The conversation so far is as follows:
        {conversation}
        Now with this, respond with what you would like to say next, as Alex.
    """
    messages.append({"role": "user", "content": user_prompt})
    response = openai.chat.completions.create(model=gpt_model, messages=messages)
    return response.choices[0].message.content

def call_gemini():
    messages = [{"role": "system", "content": gemini_system}]
    conversation = ""
    for gpt, gemini_message, ollama_message in zip(gpt_messages, gemini_messages, ollama_messages):
        if gpt and gemini and ollama:
            conversation = conversation + f"### GPT:\n{gpt}\n"
            conversation = conversation + f"### Gemini:\n{gemini_message}\n"
            conversation = conversation + f"### Ollama:\n{ollama_message}\n"
    user_prompt = f"""
        You are Blake, in a conversation with Alex and Charlie.
        The conversation so far is as follows:
        {conversation}
        Alex: {gpt_messages[-1]}
        Now with this, respond with what you would like to say next, as Blake.
    """
    messages.append({"role": "user", "content": user_prompt})
    response = gemini.chat.completions.create(model=gemini_model, messages=messages)
    return response.choices[0].message.content

def call_ollama():
    messages = [{"role": "system", "content": ollama_system}]
    conversation = ""
    for gpt, gemini_message, ollama_message in zip(gpt_messages, gemini_messages, ollama_messages):
        if gpt and gemini and ollama:
            conversation = conversation + f"### GPT:\n{gpt}\n"
            conversation = conversation + f"### Gemini:\n{gemini_message}\n"
            conversation = conversation + f"### Ollama:\n{ollama_message}\n"
    user_prompt = f"""
        You are Charlie, in a conversation with Alex and Blake.
        The conversation so far is as follows:
        {conversation}
        Alex: {gpt_messages[-1]}
        Blake: {gemini_messages[-1]}
        Now with this, respond with what you would like to say next, as Charlie.
    """
    messages.append({"role": "user", "content": user_prompt})
    response = gemini.chat.completions.create(model=gemini_model, messages=messages)
    return response.choices[0].message.content


In [ ]:
# Execute chats
display(Markdown(f"### GPT:\n{gpt_messages[0]}\n"))
display(Markdown(f"### Gemini:\n{gemini_messages[0]}\n"))
display(Markdown(f"### Ollama:\n{ollama_messages[0]}\n"))

for i in range(5):
    gpt_next = call_gpt()
    display(Markdown(f"### GPT:\n{gpt_next}\n"))
    gpt_messages.append(gpt_next)
    
    gemini_next = call_gemini()
    display(Markdown(f"### Gemini:\n{gemini_next}\n"))
    gemini_messages.append(gemini_next)

    ollama_next = call_ollama()
    display(Markdown(f"### Ollama:\n{ollama_next}\n"))
    ollama_messages.append(ollama_next)
